In [1]:
from pathlib import Path

import cv2
import shutil

import numpy as np
from PIL import Image, ImageDraw
from shapely.geometry import Polygon

In [2]:
zone_mapping = {
    0: (255, 0, 0),
    1: (255, 255, 0),
    2: (0, 0, 255)
}

def mask_to_polygons(mask_path):
    mask = np.asarray(Image.open(mask_path).convert('RGB'))
    res = {}
    if mask.sum() < 10:
        return None, None

    for id, color in zone_mapping.items():
        if id == 0:
            mask_single = mask.sum(axis=-1) > 0
        # if id == 1:
        #     mask_single = np.all(mask[..., :2] == color[:2], axis=-1)
        else:
            mask_single = np.all(mask == color, axis=-1)

        if not mask_single.sum():
            continue

        contours, _ = cv2.findContours(mask_single.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        polygons = []
        normalized_polygons = []

        for contour in contours:
            try:
                polygon = contour.reshape(-1, 2).tolist()
                # normalized_polygon = [[round(coord[0] / mask.shape[1] , 4), round(coord[1] / mask.shape[0] , 4)] for coord in polygon]
                polygon_shapely = Polygon(polygon)
                simplified_polygon = polygon_shapely.simplify(0.99, preserve_topology=True)
                if len(simplified_polygon.exterior.coords) < 4:
                    continue
                simplified_polygon_norm = [[round(coord[0] / mask.shape[1] , 4), round(coord[1] / mask.shape[0] , 4)] for coord in simplified_polygon.exterior.coords]
                polygons.append(simplified_polygon)
                normalized_polygons.append(Polygon(simplified_polygon_norm))
            except Exception as e:
                pass
        
        res[id] = normalized_polygons

    return res

In [11]:
dataset_name = '13.10'
dataset_name_wodot = dataset_name.replace('.', '-')

root_dir = Path(f'/mnt/c/Users/egorn/Desktop/WDP/dataset/yolo_det_dataset_10_sep/Разметка/{dataset_name}/')
mask_dir = root_dir / f'{dataset_name} редакт'
new_root = root_dir / 'yolo_ready'
new_masks = new_root / 'masks'
new_origs = new_root / 'original'
txt_dir = new_root / 'txt_poly'


txt_dir.mkdir(parents=True, exist_ok=True)
new_masks.mkdir(parents=True, exist_ok=True)
new_origs.mkdir(parents=True, exist_ok=True)

In [12]:
orig_dir = root_dir / 'pics/original_frames'

for mask_path in mask_dir.glob('*.png'):
    shutil.copy(mask_path, new_masks / f'{dataset_name_wodot}_{mask_path.stem}.png')
    orig_path = orig_dir / f'{mask_path.stem.split("_mask")[0]}_original.png'
    if not orig_path.exists():
        raise FileNotFoundError
    orig_path_new = new_origs / f'{dataset_name_wodot}_{orig_path.stem.split("_original")[0]}.png'
    shutil.copy(orig_path, orig_path_new)

In [13]:
for mask_path in new_masks.glob('*.png'):
    res = mask_to_polygons(mask_path)
    txt_path = txt_dir / (mask_path.name.split('_mask')[0] + '.txt')
    final_str = ''
    for class_idx, class_elems in res.items():
        for elem in class_elems:
            points_str = f'{class_idx}'
            for point in elem.exterior.coords:
                x, y = point
                points_str += f' {x} {y}'
            final_str += points_str + '\n' 
    
    txt_path.write_text(final_str)

In [14]:
# for orig_path in new_origs.glob('*.png'):
#     shutil.move(orig_path, new_origs / f'{orig_path.stem.split("_original")[0]}.png')

In [24]:
im = Image.new('L', (640, 480), 0)
draw = ImageDraw.Draw(im)

for point in res[2][0].exterior.coords:    
    x, y = point
    # points_str += f' {x} {y}'
    draw.ellipse((x * im.size[0] - 1, y * im.size[1] - 1 ,  x * im.size[0] + 1,  y * im.size[1] + 1), fill=255)

In [ ]:
for 